# Interactive Webmap — UHI & Change Surfaces, All Layers

**Library:** [`sitex`](../sitex) · **Original, fully-documented version:** [`04-VIZ-InteractiveWebmap.ipynb`](../documentations/04-VIZ-InteractiveWebmap.ipynb)

Consolidates every raster layer from `04-VIZ-Building_UHI_Overlays.ipynb` into a
single interactive folium map: LST, NDVI, HEI (Landsat, 30 m) and NDVI/NDBI change
2017→2026 (Sentinel-2, 10 m), each a toggleable overlay — plus a plain building-
footprints outline layer on top.

An earlier version of this map also sampled a raster value per building centroid to
draw a "crisp" per-building vector layer. That layer has been dropped: at 30 m/10 m
source resolution, many building centroids land in the very same pixel, so the
per-building colouring read as building-level precision the underlying data doesn't
actually have. Keeping the map to raster overlays + a plain footprint outline avoids
that false impression.

In [1]:
# One-time setup, if `sitex` isn't already installed in this kernel:
# %pip install "sitex[viz] @ git+https://github.com/ArchiColab/sitex.git"

import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print("Running in:", "Google Colab" if IN_COLAB else "Local environment")

if IN_COLAB:
    %pip install "sitex[viz] @ git+https://github.com/ArchiColab/sitex.git"

    from google.colab import drive
    drive.mount("/content/gdrive")

    DATA_DIR = Path("/content/gdrive/MyDrive/Colab_Outputs")
    OUTPUT_DIR = Path("/content/gdrive/MyDrive/SiteX_Outputs")

    if not (DATA_DIR / "overture").is_dir():
        raise FileNotFoundError(
            f"{DATA_DIR} not found (or incomplete) in your Google Drive. "
            "Before the workshop: download the workshop data zip and unzip it into "
            "My Drive/Colab_Outputs -- see the workshop setup instructions."
        )
else:
    DATA_DIR = Path("..") / "data"
    OUTPUT_DIR = Path("..") / "outputs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import geopandas as gpd

from sitex.core.config import CityConfig
from sitex.viz import interactive_webmap as iw
from sitex.viz import raster_scenes as rs

## 1. Configuration & inputs

In [2]:
city = CityConfig(
    place_name="Phường Pleiku, Gia Lai, Vietnam",
    local_lat=13.9833,
    local_lon=108.0000,
    slug="phường_pleiku",
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
)
HEI_PRIMARY = "50_50"
YEAR_EARLY, YEAR_RECENT = 2017, 2026

scene_dir = rs.find_latest_uhi_scene(city.data_dir / "landsat" / "output", HEI_PRIMARY)
uhi_layers = rs.load_uhi_layer_rasters(scene_dir, HEI_PRIMARY)
change_layers = rs.load_change_rasters(city.output_dir, YEAR_EARLY, YEAR_RECENT)
layers = {**uhi_layers, **change_layers}

LAYER_TITLES = {
    "lst": "Land Surface Temperature", "ndvi": "NDVI", "hei": "Heat Exposure Index",
    "ndvi_change": f"NDVI change ({YEAR_EARLY}\u2192{YEAR_RECENT})",
    "ndbi_change": f"NDBI change ({YEAR_EARLY}\u2192{YEAR_RECENT})",
}
print(f"Using Landsat scene: {scene_dir.name}")

Using Landsat scene: LC09_L2SP_124050_20260429_02_T1


## 2. Reproject to WGS84 & colourize

In [3]:
overlays = {}
for key, spec in layers.items():
    reproj_arr, bounds = iw.reproject_array_to_wgs84(spec["arr"], spec["transform"], spec["crs"])
    rgba = iw.colorize_to_rgba(reproj_arr, spec["cmap"], spec["vmin"], spec["vmax"])
    overlays[LAYER_TITLES[key]] = (rgba, bounds)
    print(f"{LAYER_TITLES[key]:<28s} -> {rgba.shape[1]}x{rgba.shape[0]} px")

Land Surface Temperature     -> 256x255 px
NDVI                         -> 256x255 px
Heat Exposure Index          -> 256x255 px
NDVI change (2017→2026)      -> 767x764 px


NDBI change (2017→2026)      -> 767x764 px


## 3. Building footprints

In [4]:
buildings = gpd.read_file(city.data_dir / "overture" / f"{city.slug}_buildings.gpkg").to_crs(f"EPSG:{city.local_epsg}")
print(f"{len(buildings):,} building footprints loaded")

SIMPLIFY_TOLERANCE_M = 0.5
buildings_web = buildings.copy()
buildings_web["geometry"] = buildings_web.geometry.simplify(SIMPLIFY_TOLERANCE_M)
buildings_web = buildings_web.to_crs("EPSG:4326")

print(f"Prepared {len(buildings_web):,} buildings for the webmap (simplified {SIMPLIFY_TOLERANCE_M} m)")

C:\Users\Maddie\anaconda3\envs\gis\lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


21,287 building footprints loaded


Prepared 21,287 buildings for the webmap (simplified 0.5 m)


## 4. Build & save the webmap

In [5]:
webmap = iw.build_uhi_change_webmap(
    (city.local_lat, city.local_lon), overlays, buildings_web,
    default_visible="Heat Exposure Index",
)

CITY_KEY = "pleiku"  # matches the original notebook's own output filename
webmap_path = city.output_dir / f"{CITY_KEY}_webmap_all_layers.html"
webmap.save(str(webmap_path))
print(f"Webmap saved to: {webmap_path}")
print(f"File size: {webmap_path.stat().st_size / 1e6:.1f} MB")

Webmap saved to: ..\..\outputs\pleiku_webmap_all_layers.html
File size: 10.1 MB


---
## Notes

- **Only the Heat Exposure Index raster is shown by default** — everything else starts
  hidden so the initial page load stays light; toggle layers from the panel (top right).
- **The map is not re-displayed inline** — with all 21k buildings and six raster
  layers, the HTML is tens of MB; open it directly in a browser instead.
- **No per-building value layer** — at this project's 30 m/10 m source resolution, a
  per-building colouring would just be restating one shared pixel's value across many
  buildings, not a real per-building measurement. The building layer here is a plain
  footprint outline; use the raster overlays for the actual surface values.